In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import scipy.stats as stats
from scipy.stats import shapiro, levene
from statsmodels.stats.multicomp import MultiComparison
import pingouin as pg
from matplotlib.patches import Patch
import scikit_posthocs as sp
from scipy.stats import chi2_contingency, chisquare, fisher_exact
from scipy.stats import chi2
import platform
import warnings
import statsmodels.api as sm
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import math
import xml.etree.ElementTree as ET
import os
from sklearn.neighbors import BallTree

warnings.filterwarnings('ignore')

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 100대명산_최종

In [ ]:
# ==========================================
# 1. 거리 및 GPX 파싱 함수
# ==========================================

def haversine(lat1, lon1, lat2, lon2):
    R_EARTH = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R_EARTH * c

def classify_course_by_ratio(file_path):
    # 1. GPX 파싱
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        # 네임스페이스 처리
        ns = {'gpx': 'http://www.topografix.com/GPX/1/1'}
        
        points = []
        for trk in root.findall('gpx:trk', ns):
            for seg in trk.findall('gpx:trkseg', ns):
                for pt in seg.findall('gpx:trkpt', ns):
                    ele_node = pt.find('gpx:ele', ns)
                    ele = float(ele_node.text) if ele_node is not None else 0.0
                    points.append((float(pt.get('lat')), float(pt.get('lon')), ele))
    except Exception as e:
        # print(f"GPX 읽기 실패: {e}")
        return None

    if not points:
        return None

    # 2. 핵심 지점 추출
    start_pt = points[0]
    end_pt = points[-1]
    
    max_point = max(points, key=lambda x: x[2])
    min_point = min(points, key=lambda x: x[2])

    start_lat, start_lon, start_ele = start_pt
    end_lat, end_lon, end_ele = end_pt
    max_ele = max_point[2]
    min_ele = min_point[2]

    # 거리 + 누적 상승/하강 계산
    total_dist = 0.0
    real_ascent = 0.0
    real_descent = 0.0

    for i in range(len(points) - 1):
        lat1, lon1, ele1 = points[i]
        lat2, lon2, ele2 = points[i+1]

        d = haversine(lat1, lon1, lat2, lon2)
        diff_ele = ele2 - ele1
        abs_de = abs(diff_ele)

        # slope 계산 (d=0 방어)
        slope = abs_de / d if d > 0 else float("inf")

        # ---------------------------
        # 1️⃣ 하드 컷: 물리적으로 불가능
        # ---------------------------
        if d < 10 and abs_de > 100:
            continue

        # ---------------------------
        # 2️⃣ 좌표 점프 제거
        # ---------------------------
        if d > 150:
            continue

        # ---------------------------
        # 3️⃣ 고도 스파이크 제거
        # ---------------------------
        if abs_de > 30 and (d < 50 or slope > 1.0):
            continue

        # ---------------------------
        # 정상 구간만 누적
        # ---------------------------
        total_dist += d

        if diff_ele > 0:
            real_ascent += diff_ele
        else:
            real_descent += abs(diff_ele)

    # 직선거리
    linear_dist = haversine(start_lat, start_lon, end_lat, end_lon)
    dist_km = total_dist / 1000.0


    # 3. 유형 판단 (비율 기반)
    total_range = max_ele - min_ele

    course_type = ""
    result_code = 0
    downhill_flag = 0

    # (1) 평지/산책
    if total_range < 30:
        if linear_dist <= total_dist * 0.15:
            course_type = "평지 원점회귀"
        else:
            course_type = "평지 이동"
        climb_up = 0.0
        come_down = 0.0
    else:
        # (2) 등산/하산
        climb_up = max_ele - start_ele
        come_down = max_ele - end_ele

        is_up_finished = False
        is_down_started = False

        if come_down < climb_up * 0.3:
            is_up_finished = True
        if climb_up < come_down * 0.3:
            is_down_started = True

        if is_up_finished:
            course_type = "편도_등산"
            result_code = 1
        elif is_down_started:
            course_type = "편도_하산"
            result_code = 1
            downhill_flag = 1
        else:
            if linear_dist <= total_dist * 0.15:
                course_type = "등하산_원점회귀"
            else:
                course_type = "등하산_횡단"

    # 하산 코스 데이터 스왑 (출발/도착 정보)
    out_start_ele = start_ele
    out_end_ele = end_ele
    out_start_lat, out_start_lon = start_lat, start_lon
    out_end_lat, out_end_lon = end_lat, end_lon

    if downhill_flag == 1:
        out_start_ele, out_end_ele = end_ele, start_ele
        out_start_lat, out_start_lon, out_end_lat, out_end_lon = (
            end_lat, end_lon, start_lat, start_lon
        )

    # 4. 예상 소요 시간 계산 (기본)
    H, U, D = 15.0, 0.1, 0.05
    def compute_time(d_km, asc, dsc):
        return (H * d_km) + (U * asc) + (D * dsc)

    time_file = compute_time(dist_km, real_ascent, real_descent)
    time_reverse = compute_time(dist_km, real_descent, real_ascent)

    # 5. 난이도 계산을 위한 보정값 (편도인 경우 왕복 가정)
    calc_dist_km = dist_km
    calc_ascent_m = real_ascent

    if result_code == 1: # 편도 등산 -> 왕복 가정
        calc_dist_km = dist_km * 2
        calc_ascent_m = real_ascent + real_descent
        final_minutes = round(time_file + time_reverse, 0)
    else:
        calc_dist_km = dist_km
        calc_ascent_m = real_ascent
        final_minutes = round(time_file, 0)

    # 시간 문자열 포맷팅
    h = int(final_minutes // 60)
    m = int(final_minutes % 60)
    time_str = f"{h}시간 {m:02d}분"

    # 6. 결과 반환 (난이도 계산을 위해 보정된 값들을 함께 넘김)
    return {
        "파일명": os.path.basename(file_path),
        "유형설명": course_type,
        "최고고도_m": round(max_ele, 1),
        "편도거리_km": round(total_dist / 1000.0, 2),
        "총거리_km": round(calc_dist_km, 2),   # 난이도 계산용 보정 거리
        "누적상승_m": round(calc_ascent_m, 1), # 난이도 계산용 보정 누적상승고도
        "예상시간_분": int(final_minutes),
        "예상시간": time_str,
        "출발_lat": out_start_lat,
        "출발_lon": out_start_lon,
        "도착_lat": out_end_lat,
        "도착_lon": out_end_lon,
    }

def analyze_all_gpx_in_folder(folder_path):
    results = []
    for root, dirs, files in os.walk(folder_path):
        for fname in files:
            if fname.lower().endswith(".gpx"):
                fpath = os.path.join(root, fname)
                row = classify_course_by_ratio(fpath)
                if row is not None:
                    results.append(row)
    
    if not results:
        print("분석할 GPX 파일이 없습니다.")
        return pd.DataFrame()

    return pd.DataFrame(results)

# ==========================================
# 2. 난이도 점수 및 티어 계산 함수
# ==========================================

METERS_TO_FEET = 3.28084
KM_TO_MILES = 0.621371

def calculate_k_mountain_score(row):
    # GPX 분석 단계에서 넘어온 보정 거리/고도를 우선 사용
    if '총거리_km' in row and row['총거리_km'] > 0:
        dist_km = row['총거리_km']
    else:
        dist_km = 0

    if '누적상승_m' in row:
        ascent_m = row['누적상승_m']
    else:
        ascent_m = 0

    dist_miles = dist_km * KM_TO_MILES
    gain_ft = ascent_m * METERS_TO_FEET
    
    # Base Score (Shenandoah)
    if gain_ft <= 0 or dist_miles <= 0:
        base_score = 0.0
    else:
        base_score = math.sqrt(2 * gain_ft * dist_miles)
    
    # Gradient Multiplier (경사도 패널티)
    if dist_km > 0:
        # 등산 구간을 전체의 절반으로 가정하고 평균 경사도 계산
        uphill_dist_km = dist_km * 0.5
        real_gradient_pct = (ascent_m / (uphill_dist_km * 1000)) * 100
    else:
        real_gradient_pct = 0
        
    if real_gradient_pct < 15: multiplier = 1.0
    elif real_gradient_pct < 25: multiplier = 1.2
    else: multiplier = 1.5
        
    return round(base_score * multiplier, 2)

# ==========================================
# 2-2. 비율(분위수) 기반 티어 컷오프 + 판정 함수
# ==========================================

TIER_LABELS = [
    ("입문", "입문"),
    ("초급", "초급1"),
    ("초급", "초급2"),
    ("초급", "초급3"),
    ("중급", "중급1"),
    ("중급", "중급2"),
    ("중급", "중급3"),
    ("상급", "상급1"),
    ("상급", "상급2"),
    ("상급", "상급3"),
    ("최상급", "최상급1"),
    ("최상급", "최상급2"),
    ("최상급", "최상급3"),
    ("초인", "초인1"),
    ("초인", "초인2"),
    ("초인", "초인3"),
    ("신", "신"),
]

TIER_QUANTILES = [
    0.05,  # 입문 상한 (5%)
    0.13,  # 초급1
    0.21,  # 초급2
    0.30,  # 초급3   -> 초급 합 25%
    0.42,  # 중급1
    0.54,  # 중급2
    0.65,  # 중급3   -> 중급 합 35% (30~65)
    0.73,  # 상급1
    0.81,  # 상급2
    0.89,  # 상급3   -> 상급 합 24% (65~89)
    0.92,  # 최상급1
    0.95,  # 최상급2
    0.97,  # 최상급3 -> 최상급 합 8% (89~97)
    0.985, # 초인1
    0.993, # 초인2
    0.997  # 초인3   -> 신은 상위 0.3%
]


def build_tier_cutoffs(scores: pd.Series) -> np.ndarray:
    """난이도점수 분포로부터 분위수 경계값(16개)을 계산."""
    s = scores.dropna().to_numpy()
    if len(s) == 0:
        return np.array([])
    return np.quantile(s, TIER_QUANTILES)

def get_tier_detail_by_cutoffs(score: float, cutoffs: np.ndarray):
    """점수와 컷오프(16개)를 받아 17단계 티어를 반환."""
    if cutoffs is None or len(cutoffs) == 0 or pd.isna(score):
        return ("입문", "입문")

    # score가 어느 구간에 속하는지 찾기
    # 예: cutoffs[0] 이하 -> 0번 라벨(입문), cutoffs[0]~cutoffs[1] -> 초급1, ... , 마지막 초과 -> 신
    idx = np.searchsorted(cutoffs, score, side="right")  # 0~16
    return TIER_LABELS[idx]

# ==========================================
# 3. 메인 실행 (통합 처리)
# ==========================================

if __name__ == "__main__":
    # 1. 경로 설정 (사용자 환경에 맞게 수정)
    gpx_folder = r'C:\Users\jiwoong\Desktop\정리\스파르타\sparta_python\최종프로젝트\100대명산'
    output_csv = '100대명산_최종.csv'

    print("Step 1: GPX 파일 분석 중...")
    df_raw = analyze_all_gpx_in_folder(gpx_folder)

    if not df_raw.empty:
        print(f"Step 2: 난이도 점수 계산 및 티어 산정 (파일 개수: {len(df_raw)}개)")
        
        # 난이도 점수 계산
        df_raw['난이도점수'] = df_raw.apply(calculate_k_mountain_score, axis=1)

        # ✅ 티어 컷오프 계산 (비율 기반)
        cutoffs = build_tier_cutoffs(df_raw["난이도점수"])

        # ✅ 티어 구분 (비율 기반)
        df_raw[['난이도', '세부난이도']] = df_raw['난이도점수'].apply(
            lambda x: pd.Series(get_tier_detail_by_cutoffs(x, cutoffs))
        )

        print("Step 3: 데이터 가공 (컬럼 분리 및 정리)")
        
        # 1. 파일명에서 확장자 제거하여 'temp_name' 생성
        df_raw['temp_name'] = df_raw['파일명'].str.rsplit('.', n=1).str[0]
        
        # 2. 산이름 추출 (파일명 형식이 '산이름_XX' 라고 가정)
        # 예: 가리산_01 -> 가리산
        df_raw['산이름'] = df_raw['temp_name'].str.split('_').str[0]
        
        # 3. 코스명 재조합 (산이름 + '_' + 뒤에 2글자)
        df_raw['코스명'] = df_raw["산이름"] + "_" + df_raw["temp_name"].str[-2:]
        
        # 4. 최종 컬럼 선택 및 순서 정렬
        final_columns = [
            '코스명', '산이름', '유형설명', 
            '최고고도_m', '누적상승_m','편도거리_km', '총거리_km', '예상시간_분', '예상시간', 
            '출발_lat', '출발_lon', '도착_lat', '도착_lon', 
            '난이도', '세부난이도', '난이도점수'
        ]
        
        # 만약 컬럼이 없을 경우를 대비해 교집합만 선택
        cols_to_use = [c for c in final_columns if c in df_raw.columns]
        df_final = df_raw[cols_to_use]

        # CSV 저장
        df_final.to_csv(output_csv, index=False, encoding="utf-8-sig")
        
        print("-" * 50)
        print(f"✅ 처리가 완료되었습니다. 결과 파일: {output_csv}")
        print("-" * 50)
        
        # 결과 미리보기
        print(df_final.head())
        
        print("\n=== 최종 티어 분포 ===")
        print(df_final['난이도'].value_counts())
    else:
        print("❌ 처리할 데이터가 없습니다.")

Step 1: GPX 파일 분석 중...
Step 2: 난이도 점수 계산 및 티어 산정 (파일 개수: 604개)
Step 3: 데이터 가공 (컬럼 분리 및 정리)
--------------------------------------------------
✅ 처리가 완료되었습니다. 결과 파일: 100대명산_최종.csv
--------------------------------------------------
       코스명   산이름    유형설명  최고고도_m  누적상승_m  편도거리_km  총거리_km  예상시간_분     예상시간  \
0   가리산_01   가리산   편도_하산   929.7   651.8     3.27    6.55     196  3시간 16분   
1   가리산_02   가리산  등하산_횡단  1037.4   760.0     6.47    6.47     205  3시간 25분   
2  가리왕산_01  가리왕산  등하산_횡단  1596.6  1160.6     9.79    9.79     321  5시간 21분   
3  가리왕산_02  가리왕산  등하산_횡단  1590.0   951.1     6.68    6.68     202  3시간 22분   
4   가야산_01   가야산   편도_등산  1416.9  1483.0     4.99    9.99     372  6시간 12분   

      출발_lat      출발_lon     도착_lat      도착_lon  난이도 세부난이도   난이도점수  
0  37.864537  127.983319  37.879128  127.967446   초급   초급2  158.32  
1  37.864428  127.984191  37.868292  127.971856   초급   초급3  169.91  
2  37.487309  128.583265  37.426318  128.570360   상급   상급2  258.28  
3  37.466601  128.538330  

# 관광인프라

In [3]:
df1_기본 = pd.read_csv('등하산_통합본.csv')
df2_기본 = pd.read_csv("왕복_all_tourist.csv")
df3_기본 = pd.read_csv("편도_통합본.csv")
df4_기본 = pd.read_csv("편도_all_tourist.csv")

In [4]:
df1_기본['temp_name'] = df1_기본['gpx_file_name'].str.rsplit('.', n=1).str[0]
df1_기본['코스명'] = df1_기본["temp_name"].str.split('_').str[0] + "_" + df1_기본["temp_name"].str[-2:]
df1_기본['distance_m'] = df1_기본['distance_m'].astype(int)

df1_기본 = df1_기본.rename(columns={
    'base_type' : '기준점',
    'place_name' : '장소명',
    'category_group_name' : '카테고리',
    'lat' : '위도',
    'lng' : '경도',
    'distance_m' : '거리(m)',
    'road_address_name' : '주소',
})

df2_기본['distance_m'] = df2_기본['distance_m'].astype(int)
df2_기본['카테고리'] = '관광명소'

df2_기본 = df2_기본.rename(columns={
    'mountain_point' : '코스명',
    'base_type' : '기준점',
    'place_name' : '장소명',
    'category' : '명소유형',
    'lat' : '위도',
    'lng' : '경도',
    'distance_m' : '거리(m)',
    'address' : '주소',
})

df3_기본['distance_m'] = df3_기본['distance_m'].astype(int)
df3_기본['기준점'] = 'start'

df3_기본 = df3_기본.rename(columns={
    'mountain' : '코스명',
    'place_name' : '장소명',
    'category' : '카테고리',
    'y' : '위도',
    'x' : '경도',
    'distance_m' : '거리(m)',
    'road_address_name' : '주소',
})

df4_기본['distance_m'] = df4_기본['distance_m'].astype(int)
df4_기본['기준점'] = 'start'
df4_기본['카테고리'] = '관광명소'

df4_기본 = df4_기본.rename(columns={
    'mountain_point' : '코스명',
    'place_name' : '장소명',
    'category' : '명소유형',
    'lat' : '위도',
    'lng' : '경도',
    'distance_m' : '거리(m)',
    'address' : '주소',
})

In [5]:
관광인프라_등하산 = pd.concat([df1_기본, df2_기본], ignore_index=True).drop_duplicates()
관광인프라_편도 = pd.concat([df3_기본, df4_기본], ignore_index=True).drop_duplicates()

관광인프라_등하산['거리가중치'] = 관광인프라_등하산['거리(m)'].map(lambda x: 1.0 if x <= 1000 else( 
                                                        0.8 if x <= 3000 else( 
                                                        0.5 )))
관광인프라_편도['거리가중치'] = 관광인프라_편도['거리(m)'].map(lambda x: 2.0 if x <= 1000 else( 
                                                        1.6 if x <= 3000 else( 
                                                        1.0 )))

관광인프라 = pd.concat([관광인프라_등하산, 관광인프라_편도], ignore_index=True).drop_duplicates()
관광인프라 = 관광인프라[관광인프라['거리(m)'] <= 5000]
관광인프라 = 관광인프라[['코스명', '기준점', '장소명', '카테고리', '명소유형', '위도', '경도', '거리(m)', '주소', '거리가중치']]

In [6]:
관광인프라.to_csv('관광인프라.csv', index = False)

# 정류장

In [7]:
# 코스 데이터
courses = pd.read_csv("100대명산_최종.csv")   # 인코딩은 상황에 맞게
# 정류장 데이터
stops   = pd.read_csv("국토교통부_전국 버스정류장 위치정보_20251031.csv", encoding="cp949")     # 예: encoding="cp949"

# 위도/경도 NaN 있는 정류장 제거
stops_clean = stops.dropna(subset=['위도', '경도']).reset_index(drop=True)

# 라디안 변환
stops_clean['lat_rad'] = np.radians(stops_clean['위도'].astype(float))
stops_clean['lon_rad'] = np.radians(stops_clean['경도'].astype(float))
stop_coords = np.vstack([stops_clean['lat_rad'], stops_clean['lon_rad']]).T

# BallTree 생성 (정류장 쪽에는 절대 NaN 없어야 함)
tree = BallTree(stop_coords, metric='haversine')

R_EARTH = 6371000  # m

def get_nearest_stop(lat, lon):
    # 코스 쪽 좌표가 NaN이면 그냥 빈 값/NaN 리턴
    if pd.isna(lat) or pd.isna(lon):
        return {
            '정류장명': None,
            '정류장_위도': np.nan,
            '정류장_경도': np.nan,
            '정류장_거리_m': np.nan
        }

    point = np.radians([[float(lat), float(lon)]])
    dist, idx = tree.query(point, k=1)
    dist_m = dist[0][0] * R_EARTH
    stop_row = stops_clean.iloc[idx[0][0]]

    return {
        '정류장명': stop_row['정류장명'],
        '정류장_위도': stop_row['위도'],
        '정류장_경도': stop_row['경도'],
        '정류장_거리_m': dist_m,
        '도시코드': stop_row['도시코드']
    }

results = []

for _, row in courses.iterrows():
    start_info = get_nearest_stop(row['출발_lat'], row['출발_lon'])
    end_info   = get_nearest_stop(row['도착_lat'], row['도착_lon'])

    merged = row.to_dict()

    # 출발 정류장 정보
    merged.update({
        '출발_정류장명':      start_info['정류장명'],
        '출발_정류장_위도':   start_info['정류장_위도'],
        '출발_정류장_경도':   start_info['정류장_경도'],
        '출발_정류장_거리_m': round(start_info['정류장_거리_m'], 1),
        '출발_도시코드': start_info['도시코드']
    })

    # 도착 정류장 정보
    merged.update({
        '도착_정류장명':      end_info['정류장명'],
        '도착_정류장_위도':   end_info['정류장_위도'],
        '도착_정류장_경도':   end_info['정류장_경도'],
        '도착_정류장_거리_m': round(end_info['정류장_거리_m'], 1),
        '도착_도시코드': end_info['도시코드']
    })

    results.append(merged)

버스정류장 = pd.DataFrame(results)

In [ ]:
# ---------------------------------------------------------
# 1. 데이터 로드
# ---------------------------------------------------------
버스정류장_추가 = pd.read_csv("버스정류장_추가.csv")

# ---------------------------------------------------------
# 2. 버스정류장_추가(수정할 데이터) 전처리
# ---------------------------------------------------------
# 버스정류장_추가의 컬럼 이름을 버스정류장의 '출발_정류장' 관련 컬럼에 맞게 변경
버스정류장_추가_rename = 버스정류장_추가.rename(columns={
                '정류장명': '출발_정류장명_new',
                '위도': '출발_정류장_위도_new',
                '경도': '출발_정류장_경도_new'
                })

# 필요한 컬럼만 선택
columns_to_merge = [
    '코스명', '출발_정류장명_new', '출발_정류장_위도_new', 
    '출발_정류장_경도_new'
]
버스정류장_추가_merge = 버스정류장_추가_rename[columns_to_merge].copy()

# ---------------------------------------------------------
# 3. 데이터 병합 (Merge) & 업데이트
# ---------------------------------------------------------
df_merged = 버스정류장.merge(버스정류장_추가_merge, on='코스명', how='left')

# 업데이트할 컬럼 목록
cols_to_update = ['출발_정류장명', '출발_정류장_위도', '출발_정류장_경도']

# combine_first를 사용하여 새 값이 있으면 덮어쓰고, 없으면(NaN) 기존 값 유지
for col in cols_to_update:
    new_col = col + '_new'
    df_merged[col] = df_merged[new_col].combine_first(df_merged[col])

# 임시 컬럼(_new) 제거
cols_to_drop_new = [col for col in df_merged.columns if col.endswith('_new')]
df_final = df_merged.drop(columns=cols_to_drop_new)

# ---------------------------------------------------------
# 4. 거리 재계산 (Haversine)
# ---------------------------------------------------------

# 출발 정류장 거리 재계산
df_final['출발_정류장_거리_m'] = round(df_final.apply(
    lambda row: haversine(
        row['출발_lat'], row['출발_lon'], 
        row['출발_정류장_위도'], row['출발_정류장_경도']
    ), axis=1
), 1)

# ---------------------------------------------------------
# 5. 최종 컬럼 정리 및 저장
# ---------------------------------------------------------

# 원하는 컬럼 순서 지정
target_columns = [
    '코스명', '유형설명', '출발_정류장명', '출발_도시코드', '출발_정류장_위도', '출발_정류장_경도', '출발_정류장_거리_m', '도착_정류장명', '도착_도시코드', '도착_정류장_위도', '도착_정류장_경도', '도착_정류장_거리_m'
]

# 해당 컬럼들만 선택하여 df_clean 생성
df_clean = df_final[target_columns]

# 확인용 출력
print("=== 최종 결과 미리보기 (상위 5개) ===")
print(df_clean.head())

# CSV 저장
df_clean.to_csv("버스정류장_최종.csv", index=False, encoding='utf-8-sig')
print("\n✅ 작업 완료: '버스정류장_최종.csv' 파일로 저장되었습니다.")

=== 최종 결과 미리보기 (상위 5개) ===
       코스명    유형설명  출발_정류장명  출발_도시코드  출발_정류장_위도   출발_정류장_경도  출발_정류장_거리_m  \
0   가리산_01   편도_하산   가리산휴양림    32310  37.861625  127.987215        471.0   
1   가리산_02  등하산_횡단   가리산휴양림    32310  37.861625  127.987215        409.4   
2  가리왕산_01  등하산_횡단     장구목이    32340  37.487273  128.583342          7.9   
3  가리왕산_02  등하산_횡단  대화버스터미널    32340  37.500217  128.452800       8421.8   
4   가야산_01   편도_등산      해인사    38400  35.792062  128.089125        499.3   

     도착_정류장명  도착_도시코드  도착_정류장_위도   도착_정류장_경도  도착_정류장_거리_m  
0     가리산휴양림    32310  37.861625  127.987215       2607.5  
1     가리산휴양림    32310  37.861625  127.987215       1538.6  
2  정선시외버스터미널    32350  37.379100  128.650533       8815.8  
3    대화버스터미널    32340  37.500217  128.452800       7466.1  
4       법전2리    37380  35.850711  128.126084       3131.2  

✅ 작업 완료: '버스정류장_최종.csv' 파일로 저장되었습니다.
